# LSTM

### 서울

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0148/0.0396 | R2: 0.9819 | MAE: 5,158 | RMSE: 7,658 | MAPE: 5.23% | MdAPE: 4.25% | RMSLE: 0.0703
Epoch  20 | Loss(T/V): 0.0137/0.0382 | R2: 0.9825 | MAE: 4,614 | RMSE: 7,534 | MAPE: 4.50% | MdAPE: 3.49% | RMSLE: 0.0608
Epoch  30 | Loss(T/V): 0.0131/0.0320 | R2: 0.9853 | MAE: 4,419 | RMSE: 6,893 | MAPE: 4.45% | MdAPE: 3.43% | RMSLE: 0.0604
Epoch  40 | Loss(T/V): 0.0119/0.0317 | R2: 0.9855 | MAE: 4,325 | RMSE: 6,861 | MAPE: 4.27% | MdAPE: 3.26% | RMSLE: 0.0584
------------------------------
FINAL TEST RESULT: R2: 0.9411 | MAE: 10,246 | RMSE: 15,637 | MAPE: 9.57% | MdAPE: 6.13% | RMSLE: 0.1271


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0075/0.0200 | R2: 0.9904 | MAE: 3,755 | RMSE: 6,342 | MAPE: 3.28% | MdAPE: 2.24% | RMSLE: 0.0513
Epoch  20 | Loss(T/V): 0.0064/0.0202 | R2: 0.9905 | MAE: 4,374 | RMSE: 6,310 | MAPE: 3.96% | MdAPE: 3.18% | RMSLE: 0.0518
Epoch  30 | Loss(T/V): 0.0063/0.0176 | R2: 0.9916 | MAE: 3,803 | RMSE: 5,912 | MAPE: 3.44% | MdAPE: 2.51% | RMSLE: 0.0478
------------------------------
FINAL TEST RESULT: R2: 0.9775 | MAE: 8,367 | RMSE: 10,009 | MAPE: 8.03% | MdAPE: 7.46% | RMSLE: 0.0880


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_서울특별시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0076/0.0209 | R2: 0.9899 | MAE: 3,791 | RMSE: 6,481 | MAPE: 3.26% | MdAPE: 2.19% | RMSLE: 0.0511
Epoch  20 | Loss(T/V): 0.0069/0.0163 | R2: 0.9922 | MAE: 3,350 | RMSE: 5,704 | MAPE: 2.98% | MdAPE: 2.07% | RMSLE: 0.0451
Epoch  30 | Loss(T/V): 0.0063/0.0174 | R2: 0.9916 | MAE: 3,558 | RMSE: 5,896 | MAPE: 3.11% | MdAPE: 2.25% | RMSLE: 0.0451
------------------------------
FINAL TEST RESULT: R2: 0.9754 | MAE: 8,789 | RMSE: 10,453 | MAPE: 8.41% | MdAPE: 7.70% | RMSLE: 0.0918


### 부산

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0192/0.0276 | R2: 0.9844 | MAE: 1,531 | RMSE: 2,524 | MAPE: 5.47% | MdAPE: 4.03% | RMSLE: 0.0734
Epoch  20 | Loss(T/V): 0.0184/0.0280 | R2: 0.9844 | MAE: 1,535 | RMSE: 2,525 | MAPE: 5.42% | MdAPE: 3.99% | RMSLE: 0.0725
Epoch  30 | Loss(T/V): 0.0170/0.0310 | R2: 0.9825 | MAE: 1,543 | RMSE: 2,670 | MAPE: 5.43% | MdAPE: 3.89% | RMSLE: 0.0743
------------------------------
FINAL TEST RESULT: R2: 0.9321 | MAE: 3,975 | RMSE: 6,970 | MAPE: 10.66% | MdAPE: 7.97% | RMSLE: 0.1435


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0139/0.0325 | R2: 0.9905 | MAE: 2,099 | RMSE: 2,949 | MAPE: 4.92% | MdAPE: 4.51% | RMSLE: 0.0575
Epoch  20 | Loss(T/V): 0.0141/0.0212 | R2: 0.9938 | MAE: 1,714 | RMSE: 2,382 | MAPE: 4.45% | MdAPE: 3.81% | RMSLE: 0.0542
Epoch  30 | Loss(T/V): 0.0126/0.0134 | R2: 0.9960 | MAE: 1,344 | RMSE: 1,905 | MAPE: 4.05% | MdAPE: 3.24% | RMSLE: 0.0517
------------------------------
FINAL TEST RESULT: R2: 0.9861 | MAE: 2,497 | RMSE: 3,606 | MAPE: 7.39% | MdAPE: 5.42% | RMSLE: 0.0925


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_부산광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0140/0.0376 | R2: 0.9884 | MAE: 2,239 | RMSE: 3,176 | MAPE: 5.20% | MdAPE: 4.66% | RMSLE: 0.0601
Epoch  20 | Loss(T/V): 0.0140/0.0233 | R2: 0.9929 | MAE: 1,822 | RMSE: 2,492 | MAPE: 4.72% | MdAPE: 4.13% | RMSLE: 0.0566
Epoch  30 | Loss(T/V): 0.0126/0.0153 | R2: 0.9953 | MAE: 1,469 | RMSE: 2,023 | MAPE: 4.45% | MdAPE: 3.76% | RMSLE: 0.0554
------------------------------
FINAL TEST RESULT: R2: 0.9860 | MAE: 2,454 | RMSE: 3,602 | MAPE: 7.11% | MdAPE: 5.37% | RMSLE: 0.0889


### 대구

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0136/0.0222 | R2: 0.9845 | MAE: 992 | RMSE: 1,681 | MAPE: 4.02% | MdAPE: 2.93% | RMSLE: 0.0593
Epoch  20 | Loss(T/V): 0.0128/0.0215 | R2: 0.9850 | MAE: 971 | RMSE: 1,654 | MAPE: 3.99% | MdAPE: 2.95% | RMSLE: 0.0580
Epoch  30 | Loss(T/V): 0.0129/0.0225 | R2: 0.9843 | MAE: 975 | RMSE: 1,691 | MAPE: 3.98% | MdAPE: 2.92% | RMSLE: 0.0572
Epoch  40 | Loss(T/V): 0.0123/0.0210 | R2: 0.9853 | MAE: 953 | RMSE: 1,636 | MAPE: 3.95% | MdAPE: 2.87% | RMSLE: 0.0567
Epoch  50 | Loss(T/V): 0.0119/0.0218 | R2: 0.9847 | MAE: 953 | RMSE: 1,668 | MAPE: 3.91% | MdAPE: 2.83% | RMSLE: 0.0563
------------------------------
FINAL TEST RESULT: R2: 0.9528 | MAE: 1,956 | RMSE: 3,018 | MAPE: 7.72% | MdAPE: 5.68% | RMSLE: 0.1031


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0103/0.0079 | R2: 0.9955 | MAE: 858 | RMSE: 1,082 | MAPE: 2.96% | MdAPE: 2.66% | RMSLE: 0.0350
Epoch  20 | Loss(T/V): 0.0095/0.0066 | R2: 0.9963 | MAE: 767 | RMSE: 982 | MAPE: 2.64% | MdAPE: 2.34% | RMSLE: 0.0319
Epoch  30 | Loss(T/V): 0.0091/0.0051 | R2: 0.9971 | MAE: 635 | RMSE: 870 | MAPE: 2.25% | MdAPE: 1.81% | RMSLE: 0.0288
------------------------------
FINAL TEST RESULT: R2: 0.9841 | MAE: 1,247 | RMSE: 1,933 | MAPE: 5.10% | MdAPE: 3.39% | RMSLE: 0.0691


In [ ]:
import os

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_대구광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0103/0.0092 | R2: 0.9945 | MAE: 917 | RMSE: 1,175 | MAPE: 3.19% | MdAPE: 2.83% | RMSLE: 0.0378
Epoch  20 | Loss(T/V): 0.0095/0.0072 | R2: 0.9957 | MAE: 801 | RMSE: 1,034 | MAPE: 2.83% | MdAPE: 2.48% | RMSLE: 0.0345
Epoch  30 | Loss(T/V): 0.0092/0.0060 | R2: 0.9965 | MAE: 683 | RMSE: 941 | MAPE: 2.45% | MdAPE: 2.03% | RMSLE: 0.0312
------------------------------
FINAL TEST RESULT: R2: 0.9850 | MAE: 1,179 | RMSE: 1,863 | MAPE: 4.77% | MdAPE: 3.07% | RMSLE: 0.0660


### 대전

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0171/0.0740 | R2: 0.9705 | MAE: 1,745 | RMSE: 3,002 | MAPE: 5.93% | MdAPE: 4.90% | RMSLE: 0.0800
Epoch  20 | Loss(T/V): 0.0161/0.0471 | R2: 0.9813 | MAE: 1,509 | RMSE: 2,392 | MAPE: 5.31% | MdAPE: 4.13% | RMSLE: 0.0708
Epoch  30 | Loss(T/V): 0.0157/0.0526 | R2: 0.9789 | MAE: 1,543 | RMSE: 2,539 | MAPE: 5.33% | MdAPE: 4.28% | RMSLE: 0.0718
Epoch  40 | Loss(T/V): 0.0154/0.0600 | R2: 0.9759 | MAE: 1,649 | RMSE: 2,713 | MAPE: 5.77% | MdAPE: 4.75% | RMSLE: 0.0783
------------------------------
FINAL TEST RESULT: R2: 0.9320 | MAE: 3,479 | RMSE: 5,217 | MAPE: 10.59% | MdAPE: 7.96% | RMSLE: 0.1342


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0129/0.0160 | R2: 0.9945 | MAE: 1,258 | RMSE: 1,691 | MAPE: 3.43% | MdAPE: 3.15% | RMSLE: 0.0411
Epoch  20 | Loss(T/V): 0.0119/0.0193 | R2: 0.9933 | MAE: 1,347 | RMSE: 1,866 | MAPE: 3.42% | MdAPE: 3.14% | RMSLE: 0.0408
Epoch  30 | Loss(T/V): 0.0117/0.0114 | R2: 0.9961 | MAE: 1,030 | RMSE: 1,433 | MAPE: 2.82% | MdAPE: 2.42% | RMSLE: 0.0349
------------------------------
FINAL TEST RESULT: R2: 0.9870 | MAE: 1,809 | RMSE: 2,493 | MAPE: 5.64% | MdAPE: 4.43% | RMSLE: 0.0714


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_대전광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0129/0.0213 | R2: 0.9924 | MAE: 1,468 | RMSE: 1,950 | MAPE: 4.02% | MdAPE: 3.80% | RMSLE: 0.0467
Epoch  20 | Loss(T/V): 0.0120/0.0247 | R2: 0.9910 | MAE: 1,555 | RMSE: 2,112 | MAPE: 4.06% | MdAPE: 3.81% | RMSLE: 0.0468
Epoch  30 | Loss(T/V): 0.0117/0.0156 | R2: 0.9943 | MAE: 1,239 | RMSE: 1,679 | MAPE: 3.44% | MdAPE: 3.10% | RMSLE: 0.0409
------------------------------
FINAL TEST RESULT: R2: 0.9866 | MAE: 1,803 | RMSE: 2,498 | MAPE: 5.62% | MdAPE: 4.34% | RMSLE: 0.0713


### 광주

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리 (결측치 행 삭제)
def load_and_prepare():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    # 결측치가 있는 행 삭제 (실거래 데이터만 남김)
    df = df.dropna(subset=FEATURES)
    
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0185/0.0280 | R2: 0.9830 | MAE: 1,022 | RMSE: 1,528 | MAPE: 5.16% | MdAPE: 4.20% | RMSLE: 0.0693
Epoch  20 | Loss(T/V): 0.0168/0.0292 | R2: 0.9822 | MAE: 1,014 | RMSE: 1,561 | MAPE: 5.06% | MdAPE: 4.04% | RMSLE: 0.0682
------------------------------
FINAL TEST RESULT: R2: 0.9060 | MAE: 2,784 | RMSE: 4,299 | MAPE: 11.14% | MdAPE: 9.54% | RMSLE: 0.1561


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 데이터 전처리
def load_and_prepare_2():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    for col in FEATURES:
        df[col] = df.groupby('sample_id')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_2()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0116/0.0127 | R2: 0.9942 | MAE: 863 | RMSE: 1,172 | MAPE: 3.57% | MdAPE: 3.35% | RMSLE: 0.0429
Epoch  20 | Loss(T/V): 0.0113/0.0047 | R2: 0.9979 | MAE: 463 | RMSE: 713 | MAPE: 2.19% | MdAPE: 1.55% | RMSLE: 0.0300
Epoch  30 | Loss(T/V): 0.0105/0.0056 | R2: 0.9974 | MAE: 514 | RMSE: 781 | MAPE: 2.15% | MdAPE: 1.69% | RMSLE: 0.0283
------------------------------
FINAL TEST RESULT: R2: 0.9856 | MAE: 1,273 | RMSE: 1,914 | MAPE: 5.23% | MdAPE: 3.60% | RMSLE: 0.0706


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(BASE_DIR)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. 환경 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_seed(seed=42):
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

def get_inverse_price(scaled_y, scaler, n_features):
    dummy = np.zeros((len(scaled_y), n_features)); dummy[:, 0] = scaled_y.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    mdape = np.median(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    rmsle = np.sqrt(np.mean(np.square(np.log1p(np.maximum(y_pred, 0)) - np.log1p(y_true))))
    return f"R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | MAPE: {mape:.2f}% | MdAPE: {mdape:.2f}% | RMSLE: {rmsle:.4f}"

# 2. 고도화된 데이터 전처리 (하이브리드)
def load_and_prepare_advanced():
    data_path = os.path.join(DATA_DIR, 'sido_광주광역시.csv')
    df = pd.read_csv(data_path, encoding='utf-8-sig')

    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    
    rt = df.groupby(['region_id', 'transaction_date'])['price'].mean().reset_index().sort_values(['region_id', 'transaction_date'])
    rt['smooth_price'] = rt.groupby('region_id')['price'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    rt['region_multiplier'] = 1 + rt.groupby('region_id')['smooth_price'].pct_change().fillna(0)
    df = pd.merge(df, rt[['region_id', 'transaction_date', 'region_multiplier']], on=['region_id', 'transaction_date'], how='left')
    
    FEATURES = ['price', 'exclusive_area', 'floor', 'school_count', 'subway_index', 'apartment_age']
    
    def apply_advanced_hybrid(group):
        group = group.sort_values('transaction_date')
        p_orig, mult = group['price'].values, group['region_multiplier'].values
        p_lin = group['price'].interpolate(method='linear', limit_direction='both').values
        actual_count = group['price'].notnull().sum()
        fidelity = actual_count / len(group)
        lin_weight = 0.8 + (0.15 * fidelity); reg_weight = 1.0 - lin_weight
        p_reg = p_orig.copy()
        for i in range(1, len(p_reg)):
            if np.isnan(p_reg[i]) and not np.isnan(p_reg[i-1]):
                p_reg[i] = p_reg[i-1] * (mult[i] if mult[i] != 0 else 1)
        p_reg = pd.Series(p_reg).fillna(pd.Series(p_lin)).values
        group['price'] = np.where(np.isnan(p_orig), (lin_weight * p_lin) + (reg_weight * p_reg), p_orig)
        for col in FEATURES[1:]: group[col] = group[col].interpolate(method='linear', limit_direction='both')
        return group
        
    df = df.groupby('sample_id').apply(apply_advanced_hybrid).reset_index(drop=True)
    le = LabelEncoder(); df['region_idx'] = le.fit_transform(df['region_id'])
    return df, le, FEATURES

df, region_le, FEATURES = load_and_prepare_advanced()

# 3. 윈도우 생성 및 샘플별 7:1:2 분할 적용
WINDOW = 12
train_windows, val_windows, test_windows = [], [], []

for _, group in df.groupby('sample_id'):
    group = group.sort_values('transaction_date')
    if len(group) <= WINDOW: continue
    
    sample_windows = []
    values = group[FEATURES].values
    reg_idxs = group['region_idx'].values
    
    for i in range(len(values) - WINDOW):
        sample_windows.append({
            'x_seq': values[i:i+WINDOW], 
            'region_idx': reg_idxs[i+WINDOW], 
            'y': values[i+WINDOW, 0]
        })
    
    n_w = len(sample_windows)
    train_windows.extend(sample_windows[:int(n_w*0.7)])
    val_windows.extend(sample_windows[int(n_w*0.7):int(n_w*0.8)])
    test_windows.extend(sample_windows[int(n_w*0.8):])

train_df = pd.DataFrame(train_windows)
val_df = pd.DataFrame(val_windows)
test_df = pd.DataFrame(test_windows)

# 4. 로더 설정 및 스케일링
n = len(train_df) + len(val_df) + len(test_df)
if n == 0:
    print("경고: 윈도우가 생성되지 않았습니다.")

scaler = StandardScaler()
# Train 데이터 기반으로 스케일러 학습
train_seq_all = np.stack(train_df['x_seq'].values)
scaler.fit(train_seq_all.reshape(-1, len(FEATURES)))

class MultiModalDataset(Dataset):
    def __init__(self, data_df, scaler, n_features):
        x_raw = np.stack(data_df['x_seq'].values); N, W, F = x_raw.shape
        self.x_seq = torch.FloatTensor(scaler.transform(x_raw.reshape(-1, F)).reshape(N, W, F))
        self.x_reg = torch.LongTensor(data_df['region_idx'].values)
        y_vals = data_df['y'].values.reshape(-1, 1); dummy = np.zeros((len(y_vals), n_features)); dummy[:, 0] = y_vals.flatten()
        self.y = torch.FloatTensor(scaler.transform(dummy)[:, 0].reshape(-1, 1))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.x_seq[idx], self.x_reg[idx], self.y[idx]

train_loader = DataLoader(MultiModalDataset(train_df, scaler, len(FEATURES)), batch_size=256, shuffle=True)
val_loader = DataLoader(MultiModalDataset(val_df, scaler, len(FEATURES)), batch_size=256)
test_loader = DataLoader(MultiModalDataset(test_df, scaler, len(FEATURES)), batch_size=256)

# 5. 모델 정의
class MultiRegionLSTM(nn.Module):
    def __init__(self, n_regions, n_features, emb_dim=16, hidden_dim=128):
        super().__init__()
        self.emb = nn.Embedding(n_regions, emb_dim); self.lstm = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(hidden_dim + emb_dim, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x_s, x_r):
        _, (h, _) = self.lstm(x_s); combined = torch.cat([h[-1], self.emb(x_r)], dim=1)
        return self.fc(combined)

model = MultiRegionLSTM(len(region_le.classes_), len(FEATURES)).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# 6. 학습
best_val_loss = float('inf'); early_stop_cnt = 0
print(f"{'='*30}\nSTART: LSTM (SAMPLE-WISE SPLIT)\n{'='*30}")

for epoch in range(1, 101):
    model.train(); t_loss = 0
    for xs, xr, y in train_loader:
        xs, xr, y = xs.to(device), xr.to(device), y.to(device)
        optimizer.zero_grad(); loss = criterion(model(xs, xr), y); loss.backward(); optimizer.step(); t_loss += loss.item()
    
    model.eval(); v_loss = 0; all_v_out, all_v_y = [], []
    with torch.no_grad():
        for vx, vr, vy in val_loader:
            vx, vr, vy = vx.to(device), vr.to(device), vy.to(device)
            v_out = model(vx, vr); v_loss += criterion(v_out, vy).item()
            all_v_out.append(v_out.cpu()); all_v_y.append(vy.cpu())
    
    avg_v_loss = v_loss / len(val_loader); scheduler.step(avg_v_loss)
    if epoch % 10 == 0:
        v_p = get_inverse_price(torch.cat(all_v_out).numpy(), scaler, len(FEATURES))
        v_a = get_inverse_price(torch.cat(all_v_y).numpy(), scaler, len(FEATURES))
        print(f"Epoch {epoch:>3} | Loss(T/V): {t_loss/len(train_loader):.4f}/{avg_v_loss:.4f} | {calculate_metrics(v_a, v_p)}")

    if avg_v_loss < best_val_loss:
        best_val_loss = avg_v_loss; torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'best_lstm_sample.pth')); early_stop_cnt = 0
    else: early_stop_cnt += 1
    if early_stop_cnt >= 20: break

# 7. 최종 결과
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_lstm_sample.pth'))); model.eval(); all_p, all_a = [], []
with torch.no_grad():
    for tx, tr, ty in test_loader:
        all_p.append(model(tx.to(device), tr.to(device)).cpu()); all_a.append(ty.cpu())
y_p = get_inverse_price(torch.cat(all_p).numpy(), scaler, len(FEATURES))
y_a = get_inverse_price(torch.cat(all_a).numpy(), scaler, len(FEATURES))
print(f"{'-'*30}\nFINAL TEST RESULT: {calculate_metrics(y_a, y_p)}\n{'='*30}")

START: LSTM (SAMPLE-WISE SPLIT)
Epoch  10 | Loss(T/V): 0.0116/0.0117 | R2: 0.9944 | MAE: 804 | RMSE: 1,125 | MAPE: 3.39% | MdAPE: 3.08% | RMSLE: 0.0416
Epoch  20 | Loss(T/V): 0.0109/0.0091 | R2: 0.9956 | MAE: 722 | RMSE: 995 | MAPE: 3.38% | MdAPE: 2.99% | RMSLE: 0.0431
Epoch  30 | Loss(T/V): 0.0105/0.0066 | R2: 0.9968 | MAE: 584 | RMSE: 847 | MAPE: 2.58% | MdAPE: 2.08% | RMSLE: 0.0335
------------------------------
FINAL TEST RESULT: R2: 0.9891 | MAE: 1,079 | RMSE: 1,653 | MAPE: 4.58% | MdAPE: 3.23% | RMSLE: 0.0642
